# Pywikibot Data Retrieval Test

Input: TSV files with entity QIDs \
Output: JSON file with entity QIDs, labels in all languages, number of sitelinks, values of all claims for  properties in all languages

Input files: https://drive.google.com/drive/folders/1G0sUD63OdXtaYBlrh6FA9uG01P9ALTbL

Properties to retrieve: 

* languages spoken, written or signed (P1412)
* place of birth (P19)
* award received (P166)
* notable works (P800)
* spouse (P26)
* occupation (P106)
* employer (P108)
* Optional: date of death (P570) - left out

Steps: \
Retrieve total number of sitelinks\
Out of those with > 10 sitelinks, select those with Wikipedia pages in all of our languages:

* German (Q188) , Zurich German (Q248682), Austrian German (Q306626)
* French (Q150), Canadian French (Q1450506)
* Italian (Q652)
* Polish (Q809)
* Hindi (Q1568 )
* English (Q1860), American English (Q7976), British English (Q7979), Canadian English(Q44676), Australian English(Q44679)
* Russian (Q7737)
* Chinese (Q7850)
* Arabic (Q13955), Egyptian Arabic (Q29919)
* Kannada (Q33673)

Notes: \
Zurich German does not have a language code: https://www.wikidata.org/wiki/Help:Wikimedia_language_codes/lists/all \
At least for Chinese (zh) and German (de) more variants exist than we initially listed. The function is_subset() therefore checks not only for zh and de, but also for all variants of these languages which follow the pattern zh-foo, or de-bar. 


In [1]:
import pywikibot
from pathlib import Path
import re
import polars as pl
import json
import logging
import sys

In [2]:
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(filename='property_retrieval.log'),
        logging.StreamHandler(sys.stdout)
    ]
)

In [7]:
def is_subset(labels, namespaces, special_labels):
    # Ensure at least one special label is present in namespaces (for Arabic: ar and arz)
    if not any(label in namespaces for label in special_labels):
        return False
    
    # Ensure for each label in labels, namespaces contains either the exact label or label followed by '-'
    for label in labels:
        if not any(ns == label or ns.startswith(f"{label}-") for ns in namespaces):
            return False
    
    return True

def get_claims(item, p_number, labels): 
    # Get claims for property associated with current item 
    claim_items = item.claims.get(p_number)

    if claim_items is not None: 
        claim_values = []
        for claim in claim_items:
            # Get labels for claims in our target languages only
            claim_labels = claim.getTarget().labels
            label_values = [claim_labels.get(label_lang) for label_lang in claim_labels if label_lang in labels] 
            claim_values.append(label_values)  
        return claim_values 
        
    return None


In [8]:
input_dir = Path("input")
output_dir = Path("output")
labels = {"en", "de", "fr", "ru", "hi", "kn", "zh", "it", "pl"}  # aber es gibt noch untervarianten der sprachen
arabic_labels = {'ar', 'arz'}
site = pywikibot.Site("wikidata", "wikidata")
repo = site.data_repository()
fails = 0
properties_dict = {
    "P1412": "Languages spoken or written",
    "P19": "Place of birth",
    "P166": "Award received",
    "P800": "Notable work",
    "P26": "Spouse",
    "P106": "Occupation",
    "P108": "Employer",
}

In [ ]:
# Process each input file individually
for file in input_dir.glob("*.tsv"):
    logging.info(f"Processing file: {file}")
    
    # Load input TSV file and remove duplicate QIDs (input contains duplicates)
    query = pl.scan_csv(file, separator="\t").with_columns(
        pl.col("creator").str.extract(r"(Q\d+)", 1).alias("creator")
    ).unique(subset=["creator"])

    # Convert polars LazyFrame into DataFrame, select column "creator"
    entities_df = query.collect()
    q_numbers = entities_df.select("creator")

    # Create entities_dict with keys for 'creator', 'num_sitelinks', 'labels_alt', and each property
    entities_dict = {
        "creator": [],
        "num_sitelinks": [],
        "labels_alt": []
    }
    
    # Add property keys from properties dictionary 
    for p_number in properties_dict:
        entities_dict[p_number] = []

    # Process each QID individually
    for q_number in q_numbers.iter_rows():
        item = pywikibot.ItemPage(repo, q_number[0])
        logging.info(f"Processing Q-number: {q_number}")
        
        try: 
            logging.info(f"Retrieving sitelinks for {q_number}")
            sitelinks = list(item.iterlinks("wikipedia"))  # Convert to list
            num_sitelinks = len(sitelinks)  # Count the number of sitelinks
            namespaces = {re.search(r"\[\[(.+?):", str(sitelink)).group(1) for sitelink in sitelinks}
        except (pywikibot.exceptions.UnknownSiteError, pywikibot.exceptions.IsRedirectPageError) as e:
            logging.error(f"Error processing {q_number}", e, exc_info=True)
            fails += 1 
            namespaces = set()

        # Check if item has at least as many sitelinks as our chosen languages, and if yes, check if sitelinks match our languages
        if num_sitelinks >= len(labels) and is_subset(labels, namespaces, arabic_labels): 
            logging.info(f"Labels are a subset of {namespaces}")
            
            labels_alt = list(set(item.labels.values()))  # remove duplicates and convert to list for storing in dictionary
            
            # Append QID, number of sitelinks, labels for entity to the dictionary
            entities_dict["creator"].append(q_number[0])
            entities_dict["num_sitelinks"].append(num_sitelinks)
            entities_dict["labels_alt"].append(labels_alt)

            # Retrieve and append claims for each property
            logging.info(f"Retrieving properties for {q_number}")
            for p_number in properties_dict:
                claim_value = get_claims(item, p_number, labels)
                entities_dict[p_number].append(claim_value)

    # Write entities_dict for current input file to JSON file
    output_file = output_dir / file.with_suffix('.json').name
    logging.info(f"Writing output to {output_file}")
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(entities_dict, f, ensure_ascii=False, indent=4)
    
    logging.info(f"Data for {file} written successfully")
    logging.info(f"Failed lookups: {fails}")

## Results 

mandarin_89.tsv
* Wall time: 2 minutes 
* Failed lookups: 1
* Number of entities with Wikipedia pages in all of our languages: 2 

russian_10000.tsv
* Wall time: 43 minutes
* Failed lookups: 18
* Number of entities with Wikipedia pages in all of our languages: 6

Output files: https://drive.google.com/drive/folders/1vX9Q7pT2BeSVR66qyaoOS7Cvzb6tLfUh

Pywikibot warnings
* "Entity schema is not supported yet": not critical, see https://www.mail-archive.com/pywikibot-bugs@lists.wikimedia.org/msg49497.html
* "Site wikipedia:be-tarask instantiated using different code "be-x-old"": unresolved